# Experimental - Find All Groups of a Given Order

In [1]:
import itertools as it
import finalg as fg

Finite Algebras:
  To see a listing of built-in example algebras, run finalg.examples.about()
  To retrieve an example algebra, use finalg.examples[INDEX]



The code below is does a brute-force search for all valid Cayley tables for a given group order.

**WARNING: Order values greater than 5 result in extremely long runtimes.**

In particular, the function `generate_all_group_tables` returns a list of all possible Cayley tables for a given order, n.

The function `tables_to_groups` uses the output list from `generate_all_group_tables` to create a list of `finalg` Groups based on the list of tables.

So, for example, for n=3, there is only one possible table/group.

For n=4 there are four valid tables/groups, the first of which is not isomorphic to any of the others, but the last three are all isomorphic to each other. So, up to isomorphism, there are only two groups of order 4, as expected.

We know that there is only one valid group of order 5. The function `generate_all_group_tables` generates 6 tables/groups for n=5, but all are isomorphic to each other, as expected.

## Code

In [2]:
def _no_conflict(p1, p2):
    """Returns True only if no element of p1 equals the corresponding element of p2."""
    return all([p1[i] != p2[i] for i in range(len(p1))])

def _no_conflicts(items):
    """Return True if each possible pair, from a list of items, has no conflicts."""
    return all(_no_conflict(combo[0], combo[1]) for combo in it.combinations(items, 2))

def _filter_out_conflicts(perms, perm, n):
    """Filter out all permutations in perms that conflict with perm,
    and don't have n as the first element."""
    nperms = [q for q in perms if q[0] == n]
    return [p for p in nperms if _no_conflict(p, perm)]

def generate_all_group_tables(order):
    """Experimental Code: Return a list of all arrays that correspond to multiplication tables for groups
    of a specific order.

    NOTE: The algorithm here is inefficient, so even very small values of 'order' will result in very
    long runtimes (e.g., order=6 ==> ~5-6 hrs runtime).
    """
    elems = list(map(str, range(order)))  # list of dummy element names
    row0 = list(range(order))
    row_candidates = [[row0]]
    for row_num in range(1, order):
        row_candidates.append(_filter_out_conflicts(it.permutations(row0), row0, row_num))
    cayley_tables = map(lambda tbl: fg.make_cayley_table(tbl, elems), list(it.product(*row_candidates)))
    return [cand for cand in cayley_tables if cand.is_associative()]

def tables_to_groups(tables, identity_name="e", elem_name="a"):
    """Given a list of Cayley tables, all the same size, turn them into a list of groups.

    NOTE: The Cayley tables expected here have the same dummy element names, so those are
    ignored and replaced with new, unique dummy names.
    """
    order = tables[0].order
    groups = []
    for j in range(len(tables)):
        gname = f"G{j}"
        desc = f"Group {j} of order {order}"
        ename = elem_name + str(j) + "_"
        elements = [identity_name + str(j)] + [f"{ename}" + str(i) for i in range(1, order)]
        groups.append(fg.make_finite_algebra(gname, desc, elements, tables[j].tolist()))
    return groups

## Examples

### Orders 3 and 4

**Order 3**: Only one valid Cayley table exists

In [3]:
tables3 = generate_all_group_tables(3)
tables3

[CayleyTable([[0, 1, 2], [1, 2, 0], [2, 0, 1]])]

**Order 4**: Four valid Cayley tables can be found, but three of them are isomorphic to each other, so **up to isomorphism** there are only two Cayley tables.

In [4]:
tables4 = generate_all_group_tables(4)
tables4

[CayleyTable([[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]]),
 CayleyTable([[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 1, 0], [3, 2, 0, 1]]),
 CayleyTable([[0, 1, 2, 3], [1, 2, 3, 0], [2, 3, 0, 1], [3, 0, 1, 2]]),
 CayleyTable([[0, 1, 2, 3], [1, 3, 0, 2], [2, 0, 3, 1], [3, 2, 1, 0]])]

In [5]:
z3 = tables_to_groups(tables3)
z3

[Group(
 'G0',
 'Group 0 of order 3',
 ('e0', 'a0_1', 'a0_2'),
 [[0, 1, 2], [1, 2, 0], [2, 0, 1]]
 )]

In [6]:
z3[0].about()


** Group **
Name: G0
Instance ID: 4909201088
Description: Group 0 of order 3
Order: 3
Identity: 'e0'
Commutative? Yes
Cyclic?: Yes
Generators: ['a0_2', 'a0_1']
Elements:
   Index   Name   Inverse  Order
      0    'e0'    'e0'       1
      1  'a0_1'  'a0_2'       3
      2  'a0_2'  'a0_1'       3
Cayley Table (showing indices):
[[0, 1, 2], [1, 2, 0], [2, 0, 1]]


'<Group:G0, ID:4909201088>'

In [7]:
groups4 = tables_to_groups(tables4)

grp = groups4[0]
for g in groups4[1:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G0 isomorphic to G1? False
G0 isomorphic to G2? False
G0 isomorphic to G3? False


In [8]:
grp = groups4[1]
for g in groups4[2:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G1 isomorphic to G2? {'a1_2': 'a2_1', 'e1': 'e2', 'a1_1': 'a2_2', 'a1_3': 'a2_3'}
G1 isomorphic to G3? {'a1_2': 'a3_1', 'e1': 'e3', 'a1_1': 'a3_3', 'a1_3': 'a3_2'}


### Order 5

**Order 5**: There are six valid Cayley tables, but all are isomorphic to each other, so *up to isomorphism* there is only one group of order 5.

In [9]:
tables5 = generate_all_group_tables(5)
len(tables5)

6

In [10]:
groups5 = tables_to_groups(tables5)
groups5

[Group(
 'G0',
 'Group 0 of order 5',
 ('e0', 'a0_1', 'a0_2', 'a0_3', 'a0_4'),
 [[0, 1, 2, 3, 4], [1, 2, 3, 4, 0], [2, 3, 4, 0, 1], [3, 4, 0, 1, 2], [4, 0, 1, 2, 3]]
 ),
 Group(
 'G1',
 'Group 1 of order 5',
 ('e1', 'a1_1', 'a1_2', 'a1_3', 'a1_4'),
 [[0, 1, 2, 3, 4], [1, 2, 4, 0, 3], [2, 4, 3, 1, 0], [3, 0, 1, 4, 2], [4, 3, 0, 2, 1]]
 ),
 Group(
 'G2',
 'Group 2 of order 5',
 ('e2', 'a2_1', 'a2_2', 'a2_3', 'a2_4'),
 [[0, 1, 2, 3, 4], [1, 3, 0, 4, 2], [2, 0, 4, 1, 3], [3, 4, 1, 2, 0], [4, 2, 3, 0, 1]]
 ),
 Group(
 'G3',
 'Group 3 of order 5',
 ('e3', 'a3_1', 'a3_2', 'a3_3', 'a3_4'),
 [[0, 1, 2, 3, 4], [1, 3, 4, 2, 0], [2, 4, 1, 0, 3], [3, 2, 0, 4, 1], [4, 0, 3, 1, 2]]
 ),
 Group(
 'G4',
 'Group 4 of order 5',
 ('e4', 'a4_1', 'a4_2', 'a4_3', 'a4_4'),
 [[0, 1, 2, 3, 4], [1, 4, 0, 2, 3], [2, 0, 3, 4, 1], [3, 2, 4, 1, 0], [4, 3, 1, 0, 2]]
 ),
 Group(
 'G5',
 'Group 5 of order 5',
 ('e5', 'a5_1', 'a5_2', 'a5_3', 'a5_4'),
 [[0, 1, 2, 3, 4], [1, 4, 3, 0, 2], [2, 3, 1, 4, 0], [3, 0, 4, 2, 1], [

In [11]:
grp = groups5[0]
for g in groups5[1:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G0 isomorphic to G1? {'a0_1': 'a1_1', 'e0': 'e1', 'a0_2': 'a1_2', 'a0_3': 'a1_4', 'a0_4': 'a1_3'}
G0 isomorphic to G2? {'a0_1': 'a2_1', 'e0': 'e2', 'a0_2': 'a2_3', 'a0_3': 'a2_4', 'a0_4': 'a2_2'}
G0 isomorphic to G3? {'a0_1': 'a3_1', 'e0': 'e3', 'a0_2': 'a3_3', 'a0_3': 'a3_2', 'a0_4': 'a3_4'}
G0 isomorphic to G4? {'a0_1': 'a4_1', 'e0': 'e4', 'a0_2': 'a4_4', 'a0_3': 'a4_3', 'a0_4': 'a4_2'}
G0 isomorphic to G5? {'a0_1': 'a5_1', 'e0': 'e5', 'a0_2': 'a5_4', 'a0_3': 'a5_2', 'a0_4': 'a5_3'}
